# Water Quality Prediction by using Machine Learning
**Forked from:** [Ramyadeveloper59/Water-Quality-Prediction-by-using-Machine-Learning](https://github.com/Ramyadeveloper59/Water-Quality-Prediction-by-using-Machine-Learning.)

**Assignment:** SYE3209 Software Construction — Week VI: Error Handling and Logging  
**Student:** Tendo Calvin | Uganda Christian University 
**Registration number:** S23B23/013
**Access number:** B24247

> This notebook preserves the original code in marked cells, then adds improved versions demonstrating proper exception handling and structured logging, per the Week VI assignment.

# Package Importing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Dataset Reading
> **Original code (preserved from fork)** — No error handling on file I/O.

In [ ]:
# ORIGINAL CODE — ANTI-PATTERN 1: No error handling on file read
# If the CSV is missing or corrupt, this crashes with a raw Python traceback.
# There is no try/except, no log message, and no meaningful error for the user.

df = pd.read_csv('water_potability.csv')
df.head()

## ✅ Improvement 1 — Logging Setup + Safe Data Loading

**Problems fixed:**
- No try/except around `pd.read_csv` → program crashes with raw traceback on missing file
- No logging infrastructure → zero visibility in production

**Fix applied:**
- Structured JSON-format logger (mirrors industry tools like ELK Stack / Datadog)
- Specific exception types caught: `FileNotFoundError`, `pd.errors.EmptyDataError`, `pd.errors.ParserError`, `UnicodeDecodeError`
- Custom `DataLoadError` exception raised — caller knows exactly what failed
- **Never** catches bare `Exception` or uses `except: pass`

In [ ]:
import logging
import sys
import json
from pathlib import Path
from typing import Optional

# ── Custom exception hierarchy ──────────────────────────────────────────────
class WaterQualityError(Exception):
    """Base exception for all water quality predictor errors."""
    pass

class DataLoadError(WaterQualityError):
    """Raised when a dataset cannot be loaded from disk."""
    pass

class DataValidationError(WaterQualityError):
    """Raised when a dataset fails schema or content checks."""
    pass

class ModelTrainingError(WaterQualityError):
    """Raised when model training fails."""
    pass

class PredictionError(WaterQualityError):
    """Raised when inference or evaluation cannot proceed."""
    pass

# ── Structured logger ───────────────────────────────────────────────────────
def configure_logging(log_level: str = 'INFO',
                       log_file: Optional[str] = None) -> logging.Logger:
    """
    Configure structured JSON-format logging to console (and optionally a file).
    This mirrors industry practice: log entries are machine-parseable and
    searchable at scale via tools like ELK Stack, Datadog, or Splunk.
    """
    logger = logging.getLogger('water_quality')
    logger.setLevel(getattr(logging, log_level.upper(), logging.INFO))
    # Prevent duplicate handlers if cell is re-run
    if logger.handlers:
        logger.handlers.clear()

    formatter = logging.Formatter(
        fmt='{"timestamp": "%(asctime)s", "level": "%(levelname)s", '
            '"function": "%(funcName)s", "message": "%(message)s"}',
        datefmt='%Y-%m-%dT%H:%M:%S'
    )
    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(formatter)
    logger.addHandler(console)

    if log_file:
        try:
            fh = logging.FileHandler(log_file, encoding='utf-8')
            fh.setFormatter(formatter)
            logger.addHandler(fh)
            logger.info(f'File logging enabled: {log_file}')
        except OSError as e:
            # Graceful degradation: fall back to console if file handler fails
            logger.warning(
                f'Could not open log file "{log_file}": {e}. '
                'Falling back to console-only logging.'
            )
    return logger

log = configure_logging(log_level='DEBUG', log_file='water_quality_assignment.log')
log.info('Logger initialised — Water Quality Prediction pipeline starting.')

# ── Safe data loader ────────────────────────────────────────────────────────
def load_data(filepath: str) -> pd.DataFrame:
    """
    Load CSV dataset with targeted exception handling.

    ORIGINAL (anti-pattern): pd.read_csv(filepath)  — no try/except
    IMPROVED: catches five specific failure modes, logs each at ERROR level,
              re-raises as domain DataLoadError so callers know what failed.
    """
    log.info(f'Loading dataset from: {filepath}')
    try:
        data = pd.read_csv(filepath)
        log.info(
            f'Dataset loaded — rows: {len(data)}, columns: {list(data.columns)}'
        )
        return data
    except FileNotFoundError:
        log.error(f'File not found: "{filepath}". Check the path and filename.')
        raise DataLoadError(f'Dataset not found: "{filepath}"')
    except PermissionError:
        log.error(f'Permission denied reading "{filepath}".')
        raise DataLoadError(f'Cannot read "{filepath}" — permission denied.')
    except pd.errors.EmptyDataError:
        log.error(f'File "{filepath}" is empty.')
        raise DataLoadError(f'Dataset file "{filepath}" contains no data.')
    except pd.errors.ParserError as e:
        log.error(f'CSV parse error in "{filepath}": {e}')
        raise DataLoadError(f'Malformed CSV "{filepath}": {e}')
    except UnicodeDecodeError as e:
        log.error(f'Encoding error in "{filepath}": {e}')
        raise DataLoadError(
            f'Cannot decode "{filepath}". Ensure it is UTF-8 encoded.'
        )

# Load data using the improved function
try:
    df = load_data('water_potability.csv')
    df.head()
except DataLoadError as e:
    log.critical(f'Aborting — data load failed: {e}')
    raise

# Dataset Shape & Info
> **Original code (preserved from fork)**

In [ ]:
# ORIGINAL CODE — plain print statements, no logging
print(df.shape)
df.info()
df.describe()

# Missing Value Analysis
> **Original code (preserved from fork)**

In [ ]:
# ORIGINAL CODE — just prints counts, no logging, no validation check
df.isnull().sum()

## ✅ Improvement 2 — Data Validation with Fail-Fast + Warning-Level Logging

**Problems fixed:**
- No schema validation — code assumes columns exist without checking
- Missing values silently ignored — no log warning, no strategy documented
- No minimum row count check — model would train on trivially small data

**Fix applied:**
- `validate_data()` checks required columns, minimum rows, and missing value ratios
- Logs at `WARNING` for non-fatal issues (missing values), `ERROR` for fatal ones
- Implements **Defensive Programming Principle #3: Fail fast when conditions are invalid**

In [ ]:
REQUIRED_FEATURES = [
    'ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
    'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity'
]
TARGET_COLUMN = 'Potability'
MIN_ROWS = 100

def validate_data(data: pd.DataFrame) -> None:
    """
    Validate dataset schema and minimum content requirements.

    Implements:
      - Defensive Programming Principle #2: Never trust external data
      - Defensive Programming Principle #3: Fail fast when invalid
    """
    log.info('Validating dataset schema and content...')

    # 1. Required columns
    all_required = REQUIRED_FEATURES + [TARGET_COLUMN]
    missing_cols = [c for c in all_required if c not in data.columns]
    if missing_cols:
        log.error(f'Missing required columns: {missing_cols}')
        raise DataValidationError(
            f'Dataset is missing expected columns: {missing_cols}. '
            f'Found: {list(data.columns)}'
        )

    # 2. Minimum row count (fail fast)
    if len(data) < MIN_ROWS:
        log.error(f'Dataset too small: {len(data)} rows (minimum: {MIN_ROWS})')
        raise DataValidationError(
            f'Need at least {MIN_ROWS} rows, got {len(data)}.'
        )

    # 3. Missing value summary — WARNING, not ERROR (handled in preprocessing)
    null_counts = data.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    if not cols_with_nulls.empty:
        log.warning(
            'Missing values detected (will be imputed in preprocessing): '
            + str(cols_with_nulls.to_dict())
        )

    # 4. Class distribution check
    dist = data[TARGET_COLUMN].value_counts(normalize=True).to_dict()
    log.info(f'Target class distribution: {dist}')
    if min(dist.values()) < 0.10:
        log.warning(
            'Severe class imbalance detected. '
            'Consider SMOTE oversampling or class_weight="balanced".'
        )

    log.info(f'Validation passed — {len(data)} rows, {len(data.columns)} columns.')

validate_data(df)

# Data Preprocessing
> **Original code (preserved from fork)** — uses `fillna(mean)` with no logging or validation.

In [ ]:
# ORIGINAL CODE — ANTI-PATTERN 2: bare except: pass (silent swallowing)
# If 'Potability' column doesn't exist, KeyError is swallowed silently.
# Function returns None and the next step crashes with a misleading error.

def preprocess_data(df):
    try:
        df['ph'] = df['ph'].fillna(df['ph'].mean())
        df['Sulfate'] = df['Sulfate'].fillna(df['Sulfate'].mean())
        df['Trihalomethanes'] = df['Trihalomethanes'].fillna(df['Trihalomethanes'].mean())
        X = df.drop(columns=['Potability'])
        y = df['Potability']
        return X, y
    except:
        pass  # ← CRITICAL anti-pattern: swallows ALL exceptions silently

# Note: calling this is risky — if it fails, X and y are None
# X, y = preprocess_data(df)

## ✅ Improvement 3 — Safe Preprocessing: Median Imputation + Specific Exception Handling

**Problems fixed:**
- `except: pass` swallows **every** exception including `SystemExit` and `KeyboardInterrupt`
- Returns `None` silently — downstream code crashes with misleading `AttributeError`
- Uses `mean()` for imputation — median is more robust for skewed water quality data
- Zero log entries — no audit trail of what was filled and with what value

**Fix applied:**
- Catches only `KeyError` and `ValueError` — the two realistic failure modes here
- Each imputed column logged at `DEBUG` level with count and median value
- Function **always** either returns valid `(X, y)` or raises `DataValidationError`
- Never returns `None`

In [ ]:
def preprocess_data_improved(data: pd.DataFrame):
    """
    Preprocess dataset: impute missing values, split features / target.

    ORIGINAL anti-pattern: except: pass  — swallows everything, returns None
    IMPROVED: catches KeyError and ValueError only; logs every imputation
              at DEBUG level; never returns None.
    """
    log.info('Starting data preprocessing...')
    try:
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            null_count = data[col].isnull().sum()
            if null_count > 0:
                # Median is more robust than mean for skewed distributions
                median_val = data[col].median()
                data[col] = data[col].fillna(median_val)
                log.debug(
                    f'Imputed {null_count} missing values in "{col}" '
                    f'with median={median_val:.4f}'
                )

        X = data[REQUIRED_FEATURES]
        y = data[TARGET_COLUMN]

        log.info(
            f'Preprocessing complete — features: {X.shape[1]}, samples: {X.shape[0]}'
        )
        return X, y

    except KeyError as e:
        # A column referenced in REQUIRED_FEATURES or TARGET_COLUMN is missing
        log.error(f'Column access error during preprocessing: {e}')
        raise DataValidationError(f'Unexpected column error: {e}')
    except ValueError as e:
        log.error(f'Value error during preprocessing: {e}')
        raise DataValidationError(f'Preprocessing failed due to invalid values: {e}')

X, y = preprocess_data_improved(df.copy())
print(f'Features shape: {X.shape}, Target shape: {y.shape}')

# Data Visualization
> **Original code (preserved from fork)**

In [ ]:
# Original visualizations — these are fine, kept as-is
plt.figure(figsize=(8,5))
df['Potability'].value_counts().plot(kind='pie', autopct='%1.1f%%',
                                      labels=['Not Potable','Potable'])
plt.title('Water Potability Distribution')
plt.ylabel('')
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

# Train/Test Split
> **Original code (preserved from fork)**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ORIGINAL CODE — no logging, no stratification, no validation of split result
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## ✅ Improvement 4 — Logged & Stratified Split with Safe Scaling

**Problems fixed:**
- No `stratify=y` → class distribution not preserved across train/test sets
- No logging → no audit trail of split sizes
- `StandardScaler` fit on full data would cause data leakage (original fit order OK but not documented)

**Fix applied:**
- `stratify=y` preserves class balance
- Split sizes logged at `INFO`
- Scaler fit **only on training data** — explicitly documented with log entry

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def split_and_scale(X, y, test_size=0.2, random_state=42):
    """
    Stratified train/test split + StandardScaler.
    Scaler is fit on training data only to prevent data leakage.
    """
    log.info(f'Splitting data — test_size={test_size}, stratified=True')
    try:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size,
            random_state=random_state,
            stratify=y   # preserves class balance in both sets
        )
        log.info(f'Split complete — train: {len(X_tr)}, test: {len(X_te)}')

        # Fit scaler on TRAIN only — prevents data leakage from test set
        log.debug('Fitting StandardScaler on training data only...')
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_te_scaled = scaler.transform(X_te)
        log.debug('Feature scaling complete.')
        return X_tr_scaled, X_te_scaled, y_tr, y_te

    except ValueError as e:
        log.error(f'Train/test split failed: {e}')
        raise ModelTrainingError(f'Could not split data: {e}')

X_train, X_test, y_train, y_test = split_and_scale(X, y)

# Model Implementation
> **Original code (preserved from fork)** — Random Forest and Naive Bayes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

# ORIGINAL CODE — ANTI-PATTERN 3: blind except Exception, returns None
# If training fails, caller gets None and predict() crashes with AttributeError

def train_rf(X_train, y_train):
    try:
        clf = RandomForestClassifier()
        clf.fit(X_train, y_train)
        return clf
    except Exception as e:
        print('Error')   # useless: no type, no context, no log level
        return None      # caller receives None — crashes at predict()

def train_nb(X_train, y_train):
    gnb = GaussianNB()
    gnb.fit(X_train, y_train)
    return gnb

# Note: if train_rf returns None, the prediction step will crash
# rf_model = train_rf(X_train, y_train)

## ✅ Improvement 5 — Safe Model Training: Specific Exceptions, Never Returns None

**Problems fixed:**
- `except Exception as e: print('Error'); return None` — three problems in one:
  1. Catches everything including `MemoryError` and `KeyboardInterrupt`
  2. `print('Error')` gives zero diagnostic information
  3. Returning `None` causes a cascade failure at `predict()` that looks unrelated

**Fix applied:**
- Catches `ValueError` (bad input) and `MemoryError` (dataset too large) specifically
- `MemoryError` logged at `CRITICAL` — requires immediate operator attention
- Functions **always** either return a fitted model or raise `ModelTrainingError`
- `class_weight='balanced'` added — handles the class imbalance flagged in validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

def train_random_forest(X_train, y_train) -> RandomForestClassifier:
    """
    Train Random Forest with proper error handling.

    ORIGINAL anti-pattern: except Exception: print('Error'); return None
    IMPROVED: catches ValueError + MemoryError; never returns None.
    """
    log.info(
        f'Training Random Forest — samples: {len(X_train)}, '
        f'features: {X_train.shape[1]}'
    )
    try:
        clf = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight='balanced'  # addresses class imbalance flagged at validation
        )
        clf.fit(X_train, y_train)
        log.info('Random Forest training complete.')
        return clf
    except ValueError as e:
        log.error(f'Invalid training data for Random Forest: {e}')
        raise ModelTrainingError(f'Random Forest training failed — bad input: {e}')
    except MemoryError:
        # CRITICAL: this requires immediate attention, not just an error log
        log.critical(
            'Out of memory during Random Forest training. '
            'Reduce n_estimators or sample the dataset.'
        )
        raise ModelTrainingError('Insufficient memory for Random Forest training.')

def train_naive_bayes(X_train, y_train) -> GaussianNB:
    """Train Gaussian Naive Bayes — second algorithm per project specification."""
    log.info('Training Gaussian Naive Bayes...')
    try:
        gnb = GaussianNB()
        gnb.fit(X_train, y_train)
        log.info('Naive Bayes training complete.')
        return gnb
    except ValueError as e:
        log.error(f'Naive Bayes training failed: {e}')
        raise ModelTrainingError(f'Naive Bayes training failed: {e}')

rf_model = train_random_forest(X_train, y_train)
nb_model = train_naive_bayes(X_train, y_train)

# Classification and Prediction
> **Original code (preserved from fork)** — no model validation before predict.

In [ ]:
# ORIGINAL CODE — ANTI-PATTERN 4: no guard against None model
# If train_rf() returned None (from anti-pattern 3), this line crashes:
# AttributeError: 'NoneType' object has no attribute 'predict'
# The real failure was in training, but the error appears here — misleading.

# rf_preds = rf_model.predict(X_test)   # would crash if model is None
# nb_preds = nb_model.predict(X_test)

## ✅ Improvement 6 — Safe Prediction with Pre-flight Model Validation

**Problems fixed:**
- `model.predict(X_test)` called without checking if model is `None`
- `AttributeError` at predict masks the real failure (training)
- `sklearn.exceptions.NotFittedError` not handled — can occur if cell execution order is wrong

**Fix applied:**
- Explicit `None` check before calling `.predict()` — logs at `ERROR`, raises `PredictionError`
- Catches `NotFittedError` separately from `ValueError`
- **Security note:** raw predictions are NOT logged — in production, prediction arrays
  can contain PII if user identifiers are embedded in features

In [ ]:
from sklearn.exceptions import NotFittedError

def predict(model, X_test, model_name: str = 'model'):
    """
    Generate predictions with pre-flight model validation.

    ORIGINAL anti-pattern: model.predict(X_test) with no None check.
    IMPROVED: validates model, catches NotFittedError and ValueError.
    SECURITY: raw predictions are not logged (potential PII exposure).
    """
    if model is None:
        log.error(
            f'Attempted prediction with None model ({model_name}). '
            'Ensure training succeeded before calling predict().'
        )
        raise PredictionError(
            f'Cannot predict — "{model_name}" model is None.'
        )

    log.info(f'Predicting with {model_name} on {len(X_test)} samples...')
    try:
        preds = model.predict(X_test)
        # Log class summary only — not raw predictions (security)
        import numpy as np
        unique, counts = np.unique(preds, return_counts=True)
        log.debug(
            f'{model_name} prediction summary: '
            + str(dict(zip(unique.tolist(), counts.tolist())))
        )
        return preds
    except NotFittedError as e:
        log.error(f'{model_name} is not fitted yet: {e}')
        raise PredictionError(f'Model "{model_name}" is not fitted: {e}')
    except ValueError as e:
        log.error(f'Prediction failed for {model_name} — input mismatch: {e}')
        raise PredictionError(f'Prediction error for "{model_name}": {e}')

rf_preds = predict(rf_model, X_test, 'RandomForest')
nb_preds = predict(nb_model, X_test, 'NaiveBayes')

# Accuracy and Evaluation
> **Original code (preserved from fork)** — bare `print`, no logging.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# ORIGINAL CODE — ANTI-PATTERN 5: print instead of logging
# No timestamp, no log level, not searchable, not machine-readable

# print('RF Accuracy:', accuracy_score(y_test, rf_preds))
# print(classification_report(y_test, rf_preds))
# print('NB Accuracy:', accuracy_score(y_test, nb_preds))
# print(classification_report(y_test, nb_preds))

## ✅ Improvement 7 — Structured Evaluation Logging

**Problems fixed:**
- `print('Accuracy:', acc)` has no timestamp, no severity level, and is not searchable
- No warning when model accuracy is unacceptably low
- Results not returned or persisted — lost after the cell runs

**Fix applied:**
- Metrics logged as structured JSON at `INFO` level — machine-parseable by ELK/Datadog
- Low-accuracy models trigger a `WARNING` log — operators can set alerts on this
- Returns a `dict` so metrics can be saved or compared downstream

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate(y_test, predictions, model_name: str) -> dict:
    """
    Evaluate model and log structured metrics at INFO level.

    ORIGINAL anti-pattern: print('Accuracy:', acc)
    IMPROVED: structured JSON log entry, low-accuracy warning, returns metrics dict.
    SECURITY: raw predictions not logged (potential PII).
    """
    log.info(f'Evaluating {model_name}...')
    try:
        acc = accuracy_score(y_test, predictions)
        report = classification_report(y_test, predictions, output_dict=True)

        # Structured log entry — machine-readable, searchable at scale
        metrics = {
            'model': model_name,
            'accuracy': round(acc, 4),
            'f1_macro': round(report.get('macro avg', {}).get('f1-score', 0), 4),
            'precision_potable': round(
                report.get('1', {}).get('precision', 0), 4),
            'recall_potable': round(
                report.get('1', {}).get('recall', 0), 4),
        }
        log.info(f'Evaluation result: {json.dumps(metrics)}')

        if acc < 0.60:
            log.warning(
                f'{model_name} accuracy is low ({acc:.1%}). '
                'Consider hyperparameter tuning, SMOTE, or feature engineering.'
            )

        # Also print human-readable report for notebook output
        print(f'\n=== {model_name} ===')
        print(classification_report(y_test, predictions,
                                    target_names=['Not Potable', 'Potable']))
        return metrics

    except ValueError as e:
        log.error(f'Evaluation failed for {model_name}: {e}')
        raise PredictionError(f'Cannot evaluate "{model_name}": {e}')

rf_metrics = evaluate(y_test, rf_preds, 'RandomForest')
nb_metrics = evaluate(y_test, nb_preds, 'NaiveBayes')

# Final Model Comparison

In [ ]:
log.info('=' * 60)
log.info('FINAL MODEL COMPARISON')
log.info(f"  Random Forest accuracy : {rf_metrics['accuracy']:.4f}")
log.info(f"  Naive Bayes accuracy   : {nb_metrics['accuracy']:.4f}")
winner = (
    'RandomForest'
    if rf_metrics['accuracy'] >= nb_metrics['accuracy']
    else 'NaiveBayes'
)
log.info(f'  Best model             : {winner}')
log.info('Pipeline complete.')
log.info('=' * 60)

---
# Assignment Summary — Week VI Error Handling & Logging

## Anti-Patterns Identified in the Original Code

| # | Location | Anti-Pattern | Risk |
|---|---|---|---|
| 1 | `pd.read_csv()` | No try/except on file I/O | HIGH — crashes on missing file |
| 2 | `preprocess_data()` | `except: pass` — silent swallow | CRITICAL — hides all failures |
| 3 | `train_rf()` | `except Exception` + `print('Error')` + `return None` | HIGH — cascade failure |
| 4 | `predict()` | No None check before `.predict()` | MEDIUM — misleading AttributeError |
| 5 | `evaluate()` | `print()` instead of `logging` | LOW — zero production observability |

## Improvements Applied

| Improvement | What Changed | Lecture Principle |
|---|---|---|
| Custom exception hierarchy | `DataLoadError`, `DataValidationError`, etc. | Meaningful error messages |
| Structured JSON logging | `{"level": ..., "function": ..., "message": ...}` | Industry logging standards |
| Specific exception types | `FileNotFoundError`, `pd.errors.ParserError`, etc. | Catch specific exceptions |
| Fail-fast validation | `validate_data()` before pipeline enters training | Fail fast principle |
| Never return None | Training functions raise or return a fitted model | Fail safely |
| Security — no PII in logs | Predictions logged as class summaries only | Logging & security |

## AI vs Human Reasoning

**What AI (Claude) identified quickly:** All five anti-patterns, the custom exception hierarchy, structured logging format, and the Never-Return-None rule.

**What human engineering judgment adds:**
- *Domain knowledge:* Median imputation is better than mean for skewed water quality data
- *Security instinct:* Recognising that prediction arrays can expose PII
- *System design:* Log file handler should degrade gracefully — observability must not itself be a single point of failure
- *Subtle ordering:* `stratify=y` must be validated after the class distribution check

> **Conclusion:** AI tools accelerate pattern detection. Human engineering judgment is required for domain-specific decisions, security implications, and system-level design that requires understanding context beyond the code itself.